# Example: Ticker-Picker DQN (Advanced, Optional)

This is an **advanced optional exercise** that solves the same combinatorial ticker-selection problem as the [Ticker-Picker Bandit](eCornell-AI-Finance-S3-Example-Optional-TickerPickerBandit-May-2026.ipynb) example, but replaces the epsilon-greedy bandit with a deep Q-network. The pitch from class: *the bandit treats every basket as an opaque arm and faces $2^{N}-1$ unrelated experts; DQN replaces that table with a state-conditioned Q-function whose shared parameters generalize across baskets, so what we learn about "AAPL + MSFT" transfers to "AAPL + GOOG".*

> __The problem in plain English__
>
> __What we are deciding.__ With $K = 22$ tickers there are $2^{22} - 1 \approx 4.2$ million non-empty subsets. The combinatorial bandit treats each subset as an arm and visits at most a few thousand of them with an exploration budget on the order of $10^{3}$ iterations. DQN reframes the same selection as a sequential MDP: the agent starts with an empty basket and adds one ticker per step until the basket reaches a fixed size $K_{\text{basket}}$. The action space at each step is the $N - |\text{basket}|$ remaining tickers, so the total action budget per episode is $K_{\text{basket}}$, not $2^{N}$.
>
> __What the agent gets back.__ Intermediate rewards are zero. The terminal reward is the same canonical Cobb-Douglas utility $U = \kappa \prod_{i \in \mathcal{S}} n_{i}^{\gamma_{i}}$ that the [`bandit_world(...)`](eCornell-AI-Finance-S3-Example-Optional-TickerPickerBandit-May-2026.ipynb) function returns, evaluated on the basket the agent has constructed by the end of the episode.
>
> __How the agent learns.__ A small Q-network $Q_{\boldsymbol{\theta}}: \{0,1\}^{N} \to \mathbb{R}^{N}$ scores every ticker as a candidate next-pick from the current basket-mask state. Training uses the canonical DQN loop from L16a: a circular replay buffer, a delayed target network synced every $C$ steps, and a single Adam step per mini-batch on the squared Bellman residual.
>
> __Which data we use.__ Same as the bandit notebook: SPY 2014-2024 training plus 2025-2026 YTD test, per-ticker prices from the extended-test OHLC dataset, decision day fixed at `BANDIT_DECISION_DATE`. The DQN trains on the same fixed-decision-day reward function so the comparison to the bandit is direct.

> __Learning Objectives:__
>
> By the end of this example, we will be able to:
> * __Train a DQN on the same combinatorial problem as the bandit:__ Wire a tiny Q-network, replay buffer, and target network into the canonical DQN loop, train on the Cobb-Douglas utility reward at a single decision day, and read the per-episode return curve to confirm convergence in a fraction of the bandit's iteration budget.
> * __Compare DQN-selected and bandit-selected baskets on the real forward path:__ Forward-walk the Cobb-Douglas rebalancing engine on each strategy's basket from `BANDIT_DECISION_DATE` through the end of the test data, and tabulate realized terminal wealth, max drawdown, and annualized Sharpe.
> * __Read the learned Q-function:__ Query the trained Q-network from the empty-basket state and check whether its per-ticker scores recover the SIM-driven preference-weight ranking $\gamma_{i}$, exposing what the network actually learned about ticker quality.

___

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via [the `Include.jl` file](./Include.jl). This activates the local [Julia](https://julialang.org) environment and loads all dependencies, including [Flux.jl](https://fluxml.ai/Flux.jl/stable/) for the Q-network.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

### Constants
The constants below set the universe + decision day (matching the bandit notebook), the DQN training schedule, and the Q-network architecture. See the inline comments for units and roles.

In [ ]:
# --- Decision-day + market context (mirrors the bandit notebook) ---
B₀                    = 10_000.0                    # starting budget (USD)
Δt                    = 1.0 / 252.0                 # trading-day step (years)
L_short               = 21                          # short EMA window (days) for sentiment
L_long                = 63                          # long EMA window (days) for sentiment
L_growth              = 10                          # EMA window for the rebalancing engine's market factor
GAIN                  = 10.0                        # gain G for the λ sentiment signal
BANDIT_DECISION_DATE  = "2025-01-02"                # the calendar day we treat as "now"
BANDIT_GM_WINDOW      = 63                          # EMA window (days) for E[g_m] fed to the reward function
BANDIT_EPSILON        = 0.1                         # ε-floor for negative-γ tickers in the CD utility
TRIGGER_MAX_DRAWDOWN  = 0.15                        # forward-engine drawdown gate
TRIGGER_MAX_TURNOVER  = 0.50                        # forward-engine turnover cap

# --- DQN environment ---
K_BASKET              = 4                           # episode length: tickers added per episode

# --- DQN training hyperparameters (mirrors L16a / L16b conventions) ---
HIDDEN                = 64                          # hidden width of the Q-network
BUFFER_CAPACITY       = 5_000                       # circular replay buffer capacity M
WARMUP                = 200                         # N_warm: no gradient updates until buffer ≥ this
MINIBATCH             = 64                          # B: mini-batch size per gradient step
TARGET_SYNC           = 50                          # C: copy main → target every TARGET_SYNC global steps
DISCOUNT              = 0.95f0                      # γ
LR                    = 1.0f-3                      # Adam learning rate
EPISODES              = 600                         # number of training episodes
EPS_FLOOR             = 0.05f0                      # floor on the t^(-1/3) ε-greedy schedule

# --- Bandit comparison budget (matches Task 1's training-sample axis) ---
BANDIT_ITERS_COMPARE  = EPISODES * K_BASKET         # 600 × 4 = 2400 transitions / iterations
BANDIT_ALPHA          = 0.1                         # constant learning rate for bandit arm-mean updates
;

### Implementation
Three notebook-local helpers live here so the task cells stay short:

* [`compute_basket_utility(mask, ctx)`](./): the canonical INFORMS-form Cobb-Douglas utility used by the bandit's [`eCornellAIFinance.bandit_world(...)`](eCornell-AI-Finance-S3-Example-Optional-TickerPickerBandit-May-2026.ipynb), repackaged to take a basket bit-mask. Returns the scalar reward $U$.
* [`ticker_picker_world(state, action, ctx)`](./): one MDP step. State is the $N$-bit basket mask, action is the index of the ticker to add (already-picked tickers are masked at action-selection time). Returns `(s_next, r, done)` with $r = 0$ on intermediate steps and $r = U$ on the terminal step.
* [`MyReplayBuffer`](./), [`make_qnet(...)`](./), [`select_action(...)`](./), [`dqn_train_step!(...)`](./), [`train_dqn(...)`](./): the canonical DQN scaffolding. The Q-network is a small MLP $Q_{\boldsymbol{\theta}}:\mathbb{R}^{N} \to \mathbb{R}^{N}$ matching the [L16a / L16b](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2026) shape; the training loop runs the canonical replay-buffer-plus-target-network update.

The code blocks below define each helper.

In [ ]:
# --- Step 1: Cobb-Douglas utility on a basket bit-mask (canonical INFORMS form) ---
# compute_basket_utility(mask, ctx) -> Float64
# Canonical INFORMS-form Cobb-Douglas utility U = κ · prod_{i∈S} n_i^{γ_i}, with the
# same allocation rule as the bandit notebook's `eCornellAIFinance.bandit_world`.

function compute_basket_utility(mask::AbstractVector, ctx)::Float64
    S = findall(==(1.0f0), Float32.(mask))
    isempty(S) && return 0.0
    γ = ctx.γ; B = ctx.B; p = ctx.prices; ε = ctx.epsilon
    N = length(γ)
    n = zeros(Float64, N)
    has_neg = any(γ[i] < 0.0 for i in S)

    if !has_neg
        γ_sum = sum(γ[i] for i in S)
        γ_sum > 0.0 || return 0.0
        for i in S
            n[i] = (γ[i] / γ_sum) * (B / p[i])
        end
    else
        B̄ = B; γ_sum = 0.0
        for i in S
            if γ[i] < 0.0
                B̄ -= ε * p[i]
                n[i] = ε
            else
                γ_sum += γ[i]
            end
        end
        if γ_sum > 0.0
            for i in S
                if γ[i] >= 0.0
                    n[i] = (γ[i] / γ_sum) * (B̄ / p[i])
                end
            end
        end
    end

    κ = has_neg ? -1.0 : 1.0
    U = κ
    for i in S
        n[i] > 0.0 && (U *= n[i]^γ[i])
    end
    return U
end;

In [ ]:
# --- Step 2: Sequential ticker-picker MDP step ---
# ticker_picker_world(state, action, ctx) -> (s_next, r, done)
# Add ticker `action` to the current basket-mask state. Intermediate reward is 0;
# terminal reward is `compute_basket_utility(...)`. Episode terminates when the
# basket reaches `ctx.K_basket` tickers.

function ticker_picker_world(state::Vector{Float32}, action::Int, ctx)
    s_next = copy(state)
    s_next[action] = 1.0f0
    basket_size = Int(round(sum(s_next)))
    done = (basket_size >= ctx.K_basket)
    r = done ? Float32(compute_basket_utility(s_next, ctx)) : 0.0f0
    return (s_next, r, done)
end;

In [ ]:
# --- Step 3: Circular replay buffer + Q-network factory ---

mutable struct MyReplayBuffer
    states::Vector{Vector{Float32}}
    actions::Vector{Int}
    rewards::Vector{Float32}
    next_states::Vector{Vector{Float32}}
    dones::Vector{Bool}
    capacity::Int
end
MyReplayBuffer(cap::Int) = MyReplayBuffer(Vector{Vector{Float32}}(), Int[], Float32[], Vector{Vector{Float32}}(), Bool[], cap)

# push_transition!(buf, s, a, r, s′, d) — append, FIFO-evict if at capacity.
function push_transition!(buf::MyReplayBuffer, s, a, r, s′, d)
    if length(buf.states) >= buf.capacity
        popfirst!(buf.states); popfirst!(buf.actions)
        popfirst!(buf.rewards); popfirst!(buf.next_states); popfirst!(buf.dones)
    end
    push!(buf.states, s); push!(buf.actions, a); push!(buf.rewards, r)
    push!(buf.next_states, s′); push!(buf.dones, d)
end

# make_qnet(N, H) -> Flux.Chain
# Two-hidden-layer MLP R^N -> R^H -> R^H -> R^N with `relu` activations,
# matching the L16b architecture (narrower hidden width because the state
# vector is N=22 rather than image-like).
make_qnet(N::Int, H::Int) = Chain(
    Dense(N, H, relu),
    Dense(H, H, relu),
    Dense(H, N),
);

In [ ]:
# --- Step 4: ε-greedy action selector with masking of already-picked tickers ---

# select_action(qnet, state, ε, N) -> Int
# ε-greedy action selection with already-picked tickers masked out: with prob ε,
# sample uniformly from the un-picked indices; otherwise argmax over qnet(state)
# restricted to the available indices.
function select_action(qnet, state::Vector{Float32}, ε::Float32, N::Int)::Int
    available = findall(==(0.0f0), state)
    isempty(available) && error("no available actions: state already full")
    if rand() <= ε
        return rand(available)
    else
        Q = vec(qnet(reshape(state, :, 1)))
        @inbounds for i in 1:N
            state[i] > 0.5f0 && (Q[i] = -Inf32)
        end
        return argmax(Q)
    end
end;

In [ ]:
# --- Step 5: One DQN gradient step on a sampled mini-batch ---

# dqn_train_step!(main, target, opt_state, buf, B, γ_disc, N) -> Float32
# Sample a mini-batch of B transitions from `buf`, compute bootstrap targets
# y_i = r_i + γ (1 - d_i) max_a' [Q'_{θ⁻}(s'_i)]_{a'} with the picked-ticker mask
# applied in s', and take one Adam step on the mean squared Bellman residual.
function dqn_train_step!(main, target, opt_state,
        buf::MyReplayBuffer, B::Int, γ_disc::Float32, N::Int)::Float32
    n = length(buf.states)
    n < B && return 0.0f0
    idx = rand(1:n, B)
    S_mat  = reduce(hcat, buf.states[idx])           # (N, B)
    Sn_mat = reduce(hcat, buf.next_states[idx])      # (N, B)
    rewards = buf.rewards[idx]                        # (B,)
    dones   = Float32.(buf.dones[idx])                # (B,)
    actions = buf.actions[idx]                        # (B,)

    # --- Step 5a: target Q-values, mask already-picked in s' ---
    Qn = target(Sn_mat)                               # (N, B)
    @inbounds for j in 1:B, i in 1:N
        Sn_mat[i, j] > 0.5f0 && (Qn[i, j] = -1.0f30)
    end
    max_Qn = vec(maximum(Qn, dims = 1))               # (B,)
    targets = rewards .+ γ_disc .* (1.0f0 .- dones) .* max_Qn

    # --- Step 5b: one-hot mask for the action taken ---
    A_onehot = zeros(Float32, N, B)
    @inbounds for j in 1:B
        A_onehot[actions[j], j] = 1.0f0
    end

    # --- Step 5c: one Adam step on MSE Bellman residual ---
    loss, grads = Flux.withgradient(main) do m
        Q = m(S_mat)
        Q_taken = vec(sum(Q .* A_onehot, dims = 1))
        Flux.mse(Q_taken, targets)
    end
    Flux.update!(opt_state, main, grads[1])
    return loss
end;

In [ ]:
# --- Step 6: Full DQN training loop (mirrors L16a algorithm) ---

# train_dqn(ctx; ...) -> (main, target, history)
# Canonical DQN loop on the ticker-picker MDP defined by `ctx`. Returns the trained
# main + target Q-networks and a NamedTuple `history` with per-episode returns,
# lengths, selected baskets, and the streaming ε schedule.
function train_dqn(ctx;
        episodes::Int, K_basket::Int, hidden::Int,
        lr::Float32, buffer_cap::Int, warmup::Int, batch_size::Int,
        sync_freq::Int, γ_disc::Float32, ε_floor::Float32)
    N = length(ctx.γ)
    main   = make_qnet(N, hidden)
    target = make_qnet(N, hidden)
    Flux.loadmodel!(target, Flux.state(main))
    opt_state = Flux.setup(Adam(lr), main)
    buf = MyReplayBuffer(buffer_cap)

    returns = Float32[]; lengths = Int[]; baskets = Vector{Vector{Int}}()
    eps_track = Float32[]
    global_step = 0

    for ep in 1:episodes
        s = zeros(Float32, N)
        ep_return = 0.0f0; ep_len = 0
        while true
            global_step += 1
            t = global_step
            ε_raw = min(1.0f0, Float32(t)^(-1/3) * (Float32(N) * log(t + 1))^(1/3))
            ε = max(ε_floor, ε_raw)
            push!(eps_track, ε)

            a = select_action(main, s, ε, N)
            s′, r, done = ticker_picker_world(s, a, ctx)
            push_transition!(buf, s, a, r, s′, done)

            ep_return += r; ep_len += 1

            if length(buf.states) >= warmup
                dqn_train_step!(main, target, opt_state, buf, batch_size, γ_disc, N)
                global_step % sync_freq == 0 && Flux.loadmodel!(target, Flux.state(main))
            end

            s = s′
            done && break
        end
        push!(returns, ep_return); push!(lengths, ep_len)
        push!(baskets, findall(==(1.0f0), s))
    end

    history = (returns = returns, lengths = lengths,
               baskets = baskets, eps_track = eps_track)
    return (main, target, history)
end;

Load Session 1 artifacts and the real OHLC datasets we will use throughout. The cell mirrors the bandit notebook's loader: it concatenates the 2014-2024 training SPY series with the 2025-2026 YTD test SPY series, finds `BANDIT_DECISION_DATE` in the combined timeline, and pre-computes the inputs to the reward function and to Task 2's forward Cobb-Douglas engine.

The cell returns:

* `my_tickers::Vector{String}`: ticker universe loaded from the Session 1 calibration.
* `sim_estimates::Vector{MySIMParameterEstimate}`, `sim_params::Dict{String,Tuple{Float64,Float64,Float64}}`: per-ticker SIM fits.
* `g_f::Float64`: continuously compounded annual risk-free rate (1/yr).
* `N::Int`: number of tickers in the universe.
* `bandit_decision_date::DateTime`: the resolved decision day from `BANDIT_DECISION_DATE`.
* `bandit_λ::Float64`, `bandit_gm_t::Float64`, `bandit_prices::Vector{Float64}`, `bandit_γ::Vector{Float64}`: snapshot inputs to the DQN reward function at the decision day.
* `forward_dates::Vector{DateTime}`, `forward_price_matrix::Matrix{Float64}`, `forward_lambda::Vector{Float64}`, `forward_gm_ema::Vector{Float64}`: forward arrays for Task 2's CD engine.

In [ ]:
(; my_tickers, sim_estimates, sim_params, g_f, N,
   bandit_decision_date, bandit_λ, bandit_gm_t, bandit_prices, bandit_γ,
   forward_dates, forward_price_matrix, forward_lambda, forward_gm_ema) = let
    using Dates

    # --- Step 1: Load S1 artifacts ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    my_tickers    = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    g_f = haskey(minvar, "g_f") ? Float64(minvar["g_f"]) : Float64(minvar["r_f"]);
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        e.ticker => (e.α, e.β, e.σ_ε) for e in sim_estimates
    );
    N = length(my_tickers);

    # --- Step 2: Real SPY history (training + extended test) ---
    train_spy = MyTrainingMarketDataSet()["dataset"]["SPY"];
    test_spy  = MyExtendedTestingMarketDataSet()["dataset"]["SPY"];
    spy_full  = vcat(train_spy, test_spy);
    sort!(spy_full, :timestamp); unique!(spy_full, :timestamp);

    target_date = Date(BANDIT_DECISION_DATE);
    bandit_decision_t = findfirst(==(target_date), Date.(spy_full.timestamp));
    isnothing(bandit_decision_t) && error("BANDIT_DECISION_DATE=$(BANDIT_DECISION_DATE) not found in SPY history");
    bandit_decision_date = spy_full.timestamp[bandit_decision_t];

    # --- Step 3: λ_t and gm_t at the decision day ---
    spy_history = spy_full.close[1:bandit_decision_t];
    ema_s_real  = compute_ema(spy_history; window = L_short);
    ema_l_real  = compute_ema(spy_history; window = L_long);
    λ_real      = compute_lambda(ema_s_real, ema_l_real; G = GAIN);
    gm_real     = compute_market_growth(spy_history; Δt = Δt);
    gm_ema_real = compute_ema(gm_real; window = BANDIT_GM_WINDOW);
    bandit_λ    = λ_real[end];
    bandit_gm_t = gm_ema_real[end];

    # --- Step 4: Per-ticker prices on the decision day ---
    test_ds = MyExtendedTestingMarketDataSet()["dataset"];
    bandit_prices = zeros(N);
    for (k, t) in enumerate(my_tickers)
        df = test_ds[t];
        i  = findfirst(==(target_date), Date.(df.timestamp));
        isnothing(i) && error("ticker $(t) missing close on $(target_date)");
        bandit_prices[k] = df.close[i];
    end

    # --- Step 5: SIM-driven preference weights γ at the decision day ---
    bandit_γ = compute_preference_weights(sim_params, my_tickers, bandit_gm_t, bandit_λ);

    # --- Step 6: Forward arrays for Task 2's CD engine ---
    ema_s_full  = compute_ema(spy_full.close; window = L_short);
    ema_l_full  = compute_ema(spy_full.close; window = L_long);
    λ_full      = compute_lambda(ema_s_full, ema_l_full; G = GAIN);
    gm_full     = compute_market_growth(spy_full.close; Δt = Δt);
    gm_ema_full = compute_ema(gm_full; window = L_growth);

    forward_dates  = spy_full.timestamp[bandit_decision_t:end];
    n_fwd          = length(forward_dates);
    forward_lambda = λ_full[bandit_decision_t:end];
    forward_gm_ema = gm_ema_full[bandit_decision_t-1:end];

    forward_price_matrix = zeros(n_fwd, N + 1);
    forward_price_matrix[:, 1] = 1:n_fwd;
    target_dates_fwd = Date.(forward_dates);
    for (k, t) in enumerate(my_tickers)
        df = test_ds[t]; df_dates = Date.(df.timestamp);
        for (day, d) in enumerate(target_dates_fwd)
            i = findfirst(==(d), df_dates);
            isnothing(i) && error("ticker $(t) missing close on $(d)");
            forward_price_matrix[day, k + 1] = df.close[i];
        end
    end

    println("Universe: $(N) tickers  ($(2^N - 1) possible subsets)")
    println("Decision day: $(Date(bandit_decision_date))")
    println("  gm_t = $(round(bandit_gm_t, digits=3)) /yr   λ = $(round(bandit_λ, digits=3))")
    println("Forward window: $(Date(forward_dates[1])) → $(Date(forward_dates[end]))  ($(n_fwd) trading days)")

    (my_tickers = my_tickers, sim_estimates = sim_estimates, sim_params = sim_params,
     g_f = g_f, N = N,
     bandit_decision_date = bandit_decision_date,
     bandit_λ = bandit_λ, bandit_gm_t = bandit_gm_t,
     bandit_prices = bandit_prices, bandit_γ = bandit_γ,
     forward_dates = forward_dates, forward_price_matrix = forward_price_matrix,
     forward_lambda = forward_lambda, forward_gm_ema = forward_gm_ema)
end;

___
## Task 1: Train DQN and Compare to the Combinatorial Bandit
In this task, we train the DQN agent on the sequential ticker-picker MDP at `BANDIT_DECISION_DATE` and run the combinatorial bandit on the same problem with a matched training-sample budget. We then plot both reward trajectories on a common transition axis to compare convergence.

> __What should we see?__
>
> The bandit explores at most `BANDIT_ITERS_COMPARE` of $2^{22}-1 \approx 4.2$ million subsets, so its reward trace stays near the per-trial sample mean and barely climbs. The DQN's per-episode return $G_{e}$ should rise as the network learns the bang-for-buck ranking and converge to a basket whose Cobb-Douglas utility is at least as high as the bandit's best-found arm, in roughly the same number of total transitions.

The code block below builds the DQN reward context, trains the agent, runs the bandit baseline, and stores them in `dqn_main::Flux.Chain`, `dqn_history::NamedTuple`, `bandit_result::`[`MyBanditResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBanditResult), and `dqn_basket::Vector{String}` (the basket Task 2 will use).

In [ ]:
(; dqn_main, dqn_history, bandit_result, dqn_basket) = let
    Random.seed!(2026)

    # --- Step 1: Build the DQN reward context (γ, prices, B at decision day) ---
    dqn_ctx = (γ = bandit_γ, prices = bandit_prices, B = B₀,
               epsilon = BANDIT_EPSILON, K_basket = K_BASKET)

    # --- Step 2: Train the DQN agent ---
    main, _target, history = train_dqn(dqn_ctx;
        episodes    = EPISODES,
        K_basket    = K_BASKET,
        hidden      = HIDDEN,
        lr          = LR,
        buffer_cap  = BUFFER_CAPACITY,
        warmup      = WARMUP,
        batch_size  = MINIBATCH,
        sync_freq   = TARGET_SYNC,
        γ_disc      = DISCOUNT,
        ε_floor     = EPS_FLOOR)

    # --- Step 3: Read off the DQN's deployed basket via greedy rollout ---
    s = zeros(Float32, N); picked = Int[]
    for _ in 1:K_BASKET
        a = select_action(main, s, 0.0f0, N)  # ε = 0 → pure greedy
        push!(picked, a); s[a] = 1.0f0
    end
    dqn_basket = my_tickers[picked]

    # --- Step 4: Run the combinatorial bandit baseline at the same decision day ---
    bandit_ctx = build(MyBanditContext, (
        tickers = my_tickers, sim_parameters = sim_params,
        prices = bandit_prices, B = B₀,
        gm_t = bandit_gm_t, lambda = bandit_λ, epsilon = BANDIT_EPSILON,
    ));
    bandit_model = build(MyEpsilonGreedyBanditModel, (
        K = N, n_iterations = BANDIT_ITERS_COMPARE, alpha = BANDIT_ALPHA,
    ));
    bandit_result = solve_bandit(bandit_model, bandit_ctx);

    # --- Step 5: Report ---
    bandit_basket = my_tickers[bandit_result.best_action .== 1]
    dqn_utility   = compute_basket_utility(s, dqn_ctx)
    println("DQN basket   ($(length(dqn_basket))): $(dqn_basket)")
    println("  greedy-rollout utility U = $(round(dqn_utility, digits=4))")
    println("Bandit basket ($(length(bandit_basket))): $(bandit_basket)")
    println("  best-arm utility U = $(round(bandit_result.best_utility, digits=4))")

    (dqn_main = main, dqn_history = history,
     bandit_result = bandit_result, dqn_basket = dqn_basket)
end;

The code block below plots both training trajectories on a common training-sample axis: bandit iterations $1, 2, \ldots, T$ and DQN episode boundaries at multiples of $K_{\text{basket}}$. A 25-point rolling mean smooths each trace; the bandit's `best_utility` and the DQN's deployed-greedy utility appear as horizontal references.

In [ ]:
let
    G_dqn = dqn_history.returns
    K = K_BASKET; n_ep = length(G_dqn)
    dqn_x = collect(K:K:K*n_ep)                             # transition index at each episode boundary

    bandit_R = bandit_result.reward_history
    bandit_x = collect(1:length(bandit_R))

    rolling(v, w) = [mean(@view v[max(1, i - w + 1):i]) for i in 1:length(v)]
    w = 25
    rolling_dqn = rolling(G_dqn, w)
    rolling_bnd = rolling(bandit_R, w)

    p = plot(bandit_x, bandit_R; c = :gray70, lw = 0.5, alpha = 0.4,
        label = "bandit per-iter reward",
        xlabel = "training samples (transitions / iterations)",
        ylabel = "reward U",
        title = "DQN vs combinatorial bandit on the same ticker-picker problem",
        titlefontsize = 11,
        bg = "gray95", framestyle = :box, fg_legend = :transparent,
        legend = :bottomright)
    plot!(p, bandit_x, rolling_bnd; c = :coral, lw = 2.0,
        label = "bandit rolling mean (w = $(w))")
    scatter!(p, dqn_x, G_dqn; ms = 2.0, msw = 0, c = :gray40, alpha = 0.4,
        label = "DQN episode return G_e")
    plot!(p, dqn_x, rolling_dqn; c = :steelblue, lw = 2.5,
        label = "DQN rolling mean (w = $(w))")

    hline!(p, [bandit_result.best_utility]; c = :coral, ls = :dash, lw = 0.8,
        alpha = 0.7, label = "bandit best-arm U")
    s_g = zeros(Float32, N)
    for t in dqn_basket
        s_g[findfirst(==(t), my_tickers)] = 1.0f0
    end
    dqn_U = compute_basket_utility(s_g,
        (γ = bandit_γ, prices = bandit_prices, B = B₀, epsilon = BANDIT_EPSILON))
    hline!(p, [dqn_U]; c = :steelblue, ls = :dash, lw = 0.8, alpha = 0.7,
        label = "DQN greedy-rollout U")

    plot(p; size = (900, 480),
        left_margin = 8Plots.mm, bottom_margin = 6Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

___
## Task 2: Forward CD Rebalancing on the DQN Basket (Real Data)
In this task, we forward-walk the Cobb-Douglas rebalancing engine from `BANDIT_DECISION_DATE` through the end of the test data on **real** prices, with the basket frozen at the DQN's greedy pick from Task 1. We compare three strategies on the same forward path:

* __DQN + CD:__ Daily Cobb-Douglas rebalancing within `dqn_basket`.
* __Bandit + CD:__ Daily Cobb-Douglas rebalancing within the bandit's best-arm basket from Task 1.
* __Full-Universe CD:__ Daily Cobb-Douglas rebalancing across all 22 tickers, no basket selection.

> __What should we see?__
>
> Whether DQN beats the bandit on this realized forward path is a function of *which* basket each picked at the decision day plus *how the market evolved* over the next 16 months. The point of the comparison is not that DQN always wins, but that the DQN basket was found with the same training budget that left the bandit short of a confident pick.

The code block below returns `forward_results::Dict{String,Vector{Float64}}` with each strategy's wealth path and `forward_summary::DataFrame` with realized terminal wealth, max drawdown, and annualized Sharpe.

In [ ]:
(; forward_results, forward_summary) = let
    n_fwd = size(forward_price_matrix, 1)
    T_trade = n_fwd - 1
    rules = build(MyTriggerRules, (
        max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER,
        rebalance_schedule = ones(Int, T_trade)
    ));

    # --- Step 1: Helper: forward CD on a basket ---
    function fwd_cd(basket::Vector{String})
        sel_idx = findall(in(basket), my_tickers)
        sel_sim = Dict(t => sim_params[t] for t in basket)
        sel_p   = forward_price_matrix[:, vcat([1], sel_idx .+ 1)]
        ctx = build(MyRebalancingContextModel, (
            B = B₀, tickers = basket, marketdata = sel_p,
            marketfactor = forward_gm_ema, sim_parameters = sel_sim,
            lambda = forward_lambda[1], Δt = Δt, epsilon = 0.1,
        ));
        res = run_rebalancing_engine(ctx, rules, forward_lambda;
            offset = 1, allocator = :cobb_douglas);
        return compute_wealth_series(res, sel_p, basket; offset = 1)
    end

    # --- Step 2: DQN + CD ---
    wealth_dqn = fwd_cd(dqn_basket)

    # --- Step 3: Bandit + CD ---
    bandit_basket = my_tickers[bandit_result.best_action .== 1]
    wealth_bandit = isempty(bandit_basket) ? fill(B₀, n_fwd) : fwd_cd(bandit_basket)

    # --- Step 4: Full-Universe CD ---
    ctx_full = build(MyRebalancingContextModel, (
        B = B₀, tickers = my_tickers, marketdata = forward_price_matrix,
        marketfactor = forward_gm_ema, sim_parameters = sim_params,
        lambda = forward_lambda[1], Δt = Δt, epsilon = 0.1,
    ));
    res_full = run_rebalancing_engine(ctx_full, rules, forward_lambda;
        offset = 1, allocator = :cobb_douglas);
    wealth_full = compute_wealth_series(res_full, forward_price_matrix, my_tickers; offset = 1)

    # --- Step 5: Realized metrics ---
    function realized_metrics(W::Vector{Float64})
        ret = diff(W) ./ W[1:end-1]
        peak = accumulate(max, W)
        max_dd = maximum((peak .- W) ./ peak)
        vol = std(ret) * sqrt(252)
        ann_ret = (W[end] / W[1])^(252.0 / (length(W) - 1)) - 1.0
        sharpe = vol > 1e-6 ? (ann_ret - g_f) / vol : 0.0
        (W_T = W[end], W_T_over_W0 = W[end] / B₀, max_dd = max_dd,
         ann_ret = ann_ret, sharpe = sharpe)
    end
    m_dqn = realized_metrics(wealth_dqn)
    m_bnd = realized_metrics(wealth_bandit)
    m_full = realized_metrics(wealth_full)

    forward_summary = DataFrame(
        "Strategy"        => ["DQN + CD", "Bandit + CD", "Full-Universe CD"],
        "Tickers"         => [length(dqn_basket), length(bandit_basket), N],
        "W_T (\$)"         => [round(m.W_T, digits = 0)              for m in (m_dqn, m_bnd, m_full)],
        "W_T / W₀"        => [round(m.W_T_over_W0, digits = 3)       for m in (m_dqn, m_bnd, m_full)],
        "Max DD (%)"      => [round(m.max_dd * 100, digits = 1)      for m in (m_dqn, m_bnd, m_full)],
        "Ann. return (%)" => [round(m.ann_ret * 100, digits = 2)     for m in (m_dqn, m_bnd, m_full)],
        "Sharpe"          => [round(m.sharpe, digits = 3)            for m in (m_dqn, m_bnd, m_full)],
    )
    println("Real forward run: $(Date(forward_dates[1])) → $(Date(forward_dates[end]))  ($(n_fwd) trading days)")
    pretty_table(forward_summary; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    forward_results = Dict(
        "DQN + CD"         => wealth_dqn,
        "Bandit + CD"      => wealth_bandit,
        "Full-Universe CD" => wealth_full,
    )
    (forward_results = forward_results, forward_summary = forward_summary)
end

The code block below plots the three forward wealth paths on a common timeline, scaled to $W/W_{0}$, with a reference line at 1.0.

In [ ]:
let
    plt = plot(xlabel = "Trading day after $(Date(bandit_decision_date))",
               ylabel = "W / W₀", title = "Forward CD Engine: DQN vs Bandit vs Full Universe",
               titlefontsize = 11, legend = :bottomright,
               bg = "gray95", framestyle = :box, fg_legend = :transparent)
    style = Dict(
        "DQN + CD"         => (color = :steelblue,  lw = 2.5),
        "Bandit + CD"      => (color = :coral,      lw = 2.0, linestyle = :dash),
        "Full-Universe CD" => (color = :darkorange, lw = 1.5, linestyle = :dot),
    )
    for label in ["DQN + CD", "Bandit + CD", "Full-Universe CD"]
        W = forward_results[label]
        plot!(plt, 0:(length(W) - 1), W ./ B₀; label = label, style[label]...)
    end
    hline!(plt, [1.0]; c = :red, ls = :dot, label = "")
    plot(plt; size = (900, 480),
        left_margin = 8Plots.mm, bottom_margin = 6Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

___
## Task 3: Read the Learned Q-Function
In this task, we read off what the trained Q-network learned by querying it from the empty-basket state and ranking the resulting per-ticker Q-values. If the network internalized the bang-for-buck criterion that drives the Cobb-Douglas reward, the highest-Q tickers from the empty state should overlap with the highest-$\gamma_{i}$ tickers at the decision day.

> __What should we see?__
>
> A bar chart sorting tickers by their Q-value from the empty basket, with each bar colored by the ticker's SIM-driven preference weight $\gamma_{i}$ at the decision day. A monotonic alignment between the two confirms that the network learned to pick high-$\gamma$ assets first, even though it never saw $\gamma_{i}$ as an explicit input feature; the basket-only state was enough because the reward function bakes $\gamma_{i}$ into the terminal payoff.

The code block below queries [`dqn_main`](./) at the empty-basket state, sorts tickers by their Q-value, and renders the bar chart.

In [ ]:
let
    s0 = zeros(Float32, N)
    Q0 = vec(dqn_main(reshape(s0, :, 1)))                       # (N,)
    order = sortperm(Q0; rev = true)
    tickers_sorted = my_tickers[order]
    Q_sorted = Q0[order]
    γ_sorted = bandit_γ[order]

    γ_norm = (γ_sorted .- minimum(γ_sorted)) ./ max(maximum(γ_sorted) - minimum(γ_sorted), 1e-9)
    bar_colors = [RGB(1.0 - g, 0.4, g) for g in γ_norm]

    p1 = bar(tickers_sorted, Q_sorted;
        c = bar_colors, msw = 0,
        xlabel = "ticker (sorted by Q from empty basket)",
        ylabel = "Q_θ(empty, ticker)",
        title = "Learned Q-values from empty basket  (color = γ_i at decision day)",
        titlefontsize = 11, legend = false, xrotation = 60,
        bg = "gray95", framestyle = :box)

    p2 = scatter(γ_sorted, Q_sorted; ms = 5, msw = 0, c = :steelblue,
        xlabel = "γ_i (SIM preference weight at decision day)",
        ylabel = "Q_θ(empty, ticker)",
        title = "Q vs γ alignment", titlefontsize = 11, legend = false,
        bg = "gray95", framestyle = :box)
    for (i, t) in enumerate(tickers_sorted)
        annotate!(p2, γ_sorted[i], Q_sorted[i],
            text(" " * t, 7, :left, :gray30))
    end

    plot(p1, p2; layout = (2, 1), size = (1000, 760),
        left_margin = 8Plots.mm, bottom_margin = 8Plots.mm,
        right_margin = 6Plots.mm, top_margin = 4Plots.mm)
end

___
## Summary
This advanced example replaced the combinatorial bandit's $2^{N}-1$-arm table with a state-conditioned Q-network that scores each ticker as a candidate next-pick, trained on the same Cobb-Douglas utility reward at the same decision day. The shared parameters across baskets generalize: what the network learns from one basket transfers to all overlapping baskets, so a few thousand transitions are enough to recover a high-quality pick on a universe where the bandit has barely scratched 0.05% of the action space.

> __Key Takeaways:__
>
> * __Function approximation breaks the table-size curse:__ The bandit's storage cost grows with the number of subsets ($2^{N}$); the Q-network's storage cost is fixed by the architecture and is independent of $N$. Generalization across baskets is automatic because the same weights score every state.
> * __Sequential basket construction shrinks the action space:__ Picking one ticker per step over $K_{\text{basket}}$ steps replaces the bandit's $2^{N}-1$-arm choice with $K_{\text{basket}}$ choices over $\le N$ actions each. The Q-network learns to score candidates conditional on the current basket, which is the structure the bandit had no way to express.
> * __The learned Q-function recovers bang-for-buck without seeing $\gamma_{i}$:__ The network's only input is the basket bit-mask, but the reward function bakes $\gamma_{i}$ into the terminal payoff. The Q-values from the empty state therefore align with the SIM-driven preference-weight ranking, which is the consistency check Task 3 visualizes.

Two natural extensions, both inside the framework we built:
* Add per-ticker features ($\gamma_{i}$, $\sigma_{\varepsilon, i}$, recent realized return) to the state vector and check whether the network deploys to a held-out decision day without retraining.
* Replace uniform sampling from the replay buffer with prioritized experience replay, sampling with probability proportional to $|y_{i} - [Q_{\boldsymbol{\theta}}(s_{i})]_{a_{i}}|$, and check whether convergence in Task 1 is faster.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use real historical data and simplified models.